In [ ]:
%pip install --upgrade pip
%pip install ipython-sql psycopg2 pandas matplotlib numpy matplotlib

In [ ]:
%load_ext sql

In [ ]:
from urllib.parse import quote_plus
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
DB_URL = "postgresql://intern_read_only:%s@api.algotrade.vn:5432/algotradeDB" % quote_plus("Bingo@0711")
%sql $DB_URL

In [ ]:
def query_command(symbol, start_date, end_date):
    return """
    select tb_max.datetime as datetime, tb_max.tickersymbol as symbol,
    tb_max.price as max,
    tb_min.price as min,
    tb_close.price as close,
    tb_open.price as open,
    tb_total.quantity as total_quantity
    from (
        select *
        from quote.max m
    ) tb_max
    inner join (
        select *
        from quote.min m
    ) tb_min
    on tb_max.tickersymbol = tb_min.tickersymbol
    and tb_max.datetime = tb_min.datetime
    inner join (
        select *
        from quote.close m
    ) tb_close
    on tb_min.tickersymbol = tb_close.tickersymbol
    and tb_min.datetime = tb_close.datetime
    inner join (
        select *
        from quote.open m
    ) tb_open
    on tb_open.tickersymbol = tb_close.tickersymbol
    and tb_open.datetime = tb_close.datetime
    inner join (
        select TO_CHAR(datetime , 'YYYY-MM-DD') as datetime , m.tickersymbol,
        max(m.quantity) as quantity
        from quote.total m
        where m.tickersymbol = '{symbol}'
        and m.datetime between '{start_date}' and '{end_date}'
        group by TO_CHAR(datetime , 'YYYY-MM-DD'), m.tickersymbol
    ) tb_total
    on tb_open.tickersymbol = tb_total.tickersymbol
    and tb_open.datetime = to_date(tb_total.datetime, 'YYYY-MM-DD')
    where tb_max.tickersymbol = '{symbol}'
    and tb_max.datetime between '{start_date}' and '{end_date}';
""".format(symbol=symbol, start_date=start_date, end_date=end_date)

In [ ]:
df = pd.DataFrame()

query_var = [
    ('VN30F2310', '2023-09-22', '2023-10-21'),
    ('VN30F2311', '2023-10-21', '2023-11-17'),
    ('VN30F2312', '2023-11-17', '2023-12-22'),
    ('VN30F2401', '2023-12-22', '2024-01-19'),
    ('VN30F2402', '2024-01-19', '2024-02-16')
]

for symbol, start_date, end_date in query_var:
    sql_query = query_command(symbol, start_date, end_date)
    result = %sql $sql_query
    df = pd.concat([df, result.DataFrame()], ignore_index=True)

# In-sample Data

In [ ]:
# Convert data type
df['datetime'] = pd.to_datetime(df['datetime'])
df['open'] = df['open'].astype(float)
df['close'] = df['close'].astype(float)
df['max'] = df['max'].astype(float)
df['min'] = df['min'].astype(float)

In [ ]:
# Plot closing price
df.plot(x='datetime', y='close', kind='line', figsize=(16, 8), title="In-sample data - Close price", xlabel="Date", ylabel="Close price", legend=False)

In [ ]:
# Plot max price
df.plot(x='datetime', y='max', kind='line', figsize=(16, 8), title="In-sample data - Max price", xlabel="Date", legend=False)

In [ ]:
# Plot min price
df.plot(x='datetime', y='min', kind='line', figsize=(16, 8), title="In-sample data - Min price", xlabel="Date", legend=False)

# Bollinger Bands

In [ ]:
# Calculate typical price for each day
df['typical_price'] = (df['max'] + df['min'] + df['close']) / 3

In [ ]:
# set number of days and standard deviations to use for Bollinger Bands
window = 10
no_of_std = 1.9

rolling_mean = df['typical_price'].rolling(window).mean()
rolling_std = df['typical_price'].rolling(window).std()

In [ ]:
# Calculate three Bollinger Bands
df['Middle Band'] = rolling_mean
df['Upper Band'] = rolling_mean + (rolling_std * no_of_std)
df['Lower Band'] = rolling_mean - (rolling_std * no_of_std)

In [ ]:
# Plot the Bollinger Bands
df[['datetime', 'Middle Band', 'Upper Band', 'Lower Band', 'close']].plot(x='datetime', y=['Middle Band', 'Upper Band', 'Lower Band', 'close'], kind='line', figsize=(16, 8), title="Bollinger Bands", xlabel="Date", legend=True)

# Backtest on Bollinger Bands


In [ ]:
df.isnull().values.any()

In [ ]:
df

## Helper Function

In [ ]:
# open position when there is trading signal. add postion to holdings.
# entry point is the price point at which a position is open.
def open_position(cur_date, position_type, entry_point, entry_points, holdings):

    #position-entry-point
    holdings.append([cur_date, position_type, entry_point])
    entry_points.append((cur_date, position_type, entry_point))
    return entry_points, holdings

In [ ]:
# close position when there is trading signal. remove postion out of holdings.
def close_positions(cur_date, cur_price, holdings, exit_points, close_long=False, close_short=False):

    total_realized_pnf = 0
    total_unrealized_pnf = 0
    is_closed = False
    reversed_position_type = lambda x: "LONG" if x=="SHORT" else "SHORT"

    for date, position_type, entry_point in holdings[:]:
        if position_type == 'LONG':
            unrealized_pnl = (cur_price - entry_point)
            if close_long:
                #position-exit-point
                holdings.remove([date, position_type, entry_point])
                total_realized_pnf += unrealized_pnl
                exit_points.append((cur_date, reversed_position_type(position_type), cur_price))
                is_closed = True

        elif position_type == 'SHORT':
            unrealized_pnl = -(cur_price - entry_point)
            if close_short:
                #position-exit-point
                holdings.remove([date, position_type, entry_point])
                total_realized_pnf += unrealized_pnl
                exit_points.append((cur_date, reversed_position_type(position_type), cur_price))
                is_closed = True

        # Close at the end of the trading contract
        if not is_closed:
            # Check if cur_date is the end of the month of the date stored in holdings
            is_end_of_month = (cur_date.day == pd.to_datetime(date + pd.offsets.MonthEnd(0)).day - 1) or (cur_date.month!=date.month)# -1 as it's counted as the end of the month in the data
            if is_end_of_month:
                #position-exit-point
                holdings.remove([date, position_type, entry_point])
                total_realized_pnf += unrealized_pnl
                exit_points.append((cur_date, reversed_position_type(position_type), cur_price))
        else:
            total_unrealized_pnf += unrealized_pnl


    return exit_points, holdings, total_realized_pnf, total_unrealized_pnf

## Backtesting

In [ ]:
# holdings is a list of on-hold positions
# holdings = [[position_type, price_point], ...]
# e.g. holdings = [['SHORT', 492.0], ['LONG', 312.0]]
holdings = []

# initial asset value in cash
asset_value = 1000

# Create empty lists to store entry and exit points
entry_points = []
exit_points = []

In [ ]:
trading_data = df.copy()

# loop through data history
for idx in df.index:
  if idx==0 or df['Middle Band'][idx-1] is None:
        continue

  cur_date = df['datetime'][idx]
  cur_price = df['typical_price'][idx]
  prev_price = df['typical_price'][idx-1]

  cur_lower = df['Lower Band'][idx]
  prev_lower = df['Lower Band'][idx-1]

  cur_upper = df['Upper Band'][idx]
  prev_upper = df['Upper Band'][idx-1]

  ## CLOSE POSITION
  threshold = 0.5 * rolling_std[idx]
  close_long = len(holdings)>0 and (cur_price <= cur_lower - threshold or (prev_price < prev_upper and cur_price >= cur_upper))
  close_short = len(holdings)>0 and (cur_price >= cur_upper + threshold or (prev_price > prev_lower and cur_price <= cur_lower))
  exit_points, holdings, total_realized_pnf, total_unrealized_pnf= close_positions(cur_date, cur_price, holdings, exit_points, close_long, close_short)

  # update asset value when position is realized
  asset_value = asset_value + total_realized_pnf

  # update asset history
  if total_realized_pnf == 0:
      trading_data.loc[idx, 'Asset'] = asset_value + total_unrealized_pnf
  else:
      trading_data.loc[idx, 'Asset'] = asset_value

  # NOTE: open 1 contract only
  if len(holdings)>0:
    continue


  ## OPEN POSITION
  # open a LONG position
  if prev_price > prev_lower and cur_price <= cur_lower:
      entry_points, holdings = open_position(cur_date, 'LONG', cur_price, entry_points, holdings)

  # open a SHORT position
  if prev_price < prev_upper and cur_price >= cur_upper:
      entry_points, holdings = open_position(cur_date, 'SHORT', cur_price, entry_points, holdings)

trading_data = trading_data.dropna()
data = trading_data.copy()

### Position entry points & Position exit points

In [ ]:
# Convert entry_points and exit_points lists into DataFrames
entry_df = pd.DataFrame(entry_points, columns=['entry_datetime', 'entry_type', 'entry_price'])
exit_df = pd.DataFrame(exit_points, columns=['exit_datetime', 'exit_type', 'exit_price'])

# Concatenate entry_df and exit_df vertically
combined_df = pd.concat([entry_df, exit_df], axis=1)

# Display combined DataFrame
combined_df

In [ ]:
# Plot the Bollinger Bands
plt.figure(figsize=(18, 9))
plt.plot(df['datetime'], df[['Middle Band', 'Upper Band', 'Lower Band', 'close']])

# Plot entry points and annotate with labels
legend=False
for x, label, y in entry_points:
    if not legend:
      plt.scatter(x, y, color='green', label='Entry Point', zorder=10, s=100, marker='s')
      legend=True
    else:
      plt.scatter(x, y, color='green', label='_nolegend_', zorder=10, s=100, marker='s')
    plt.annotate(label, (x, y), textcoords="offset points", xytext=(0,10), ha='center')

# Plot exit points and annotate with labels
legend=False
for x, label, y in exit_points:
    if not legend:
      plt.scatter(x, y, color='red', label='Exit Point', zorder=10, s=100, marker='^')
      legend=True
    else:
      plt.scatter(x, y, color='red', label='_nolegend_', zorder=10, s=100, marker='^')
    plt.annotate(label, (x, y), textcoords="offset points", xytext=(0,10), ha='center')

plt.title('Bollinger Bands with Entry and Exit Points')
plt.xlabel('Date')
plt.ylabel('Price')
plt.legend()
plt.grid(True)
plt.show()



## Accumulated Return

In [ ]:
trading_data['Asset']

In [ ]:
# plot asset value over time

data[['Asset', 'datetime']].plot(x='datetime', y='Asset',kind='line', figsize=(8, 4), title='asset')
plt.gca().spines[['top', 'right']].set_visible(False)

## Maximum Drawdown (MDD)

In [ ]:
# for each day, calculate the peak of asset value since inception
data['peak'] = data.apply(lambda row: data.loc[:row.name, 'Asset'].max(), axis=1)

# for each day, calculate asset drawdown
data['drawdown'] = data['Asset']/data['peak'] - 1

# max drawdown is the most negative value
mdd = data['drawdown'].min() * 100

In [ ]:
mdd

## Sharpe ratio

In [ ]:
daily_return = data['Asset'][1:].to_numpy() / data['Asset'][:-1].to_numpy() - 1

# plot daily return
x = list(range(len(daily_return)))
plt.plot(x, daily_return)
plt.title(label="daily return")
plt.show()

In [ ]:
trading_days_per_year = 252
risk_free_rate = 0.03 # e.g. goverment bonds interest is 3% per year

# annual standard deviation
annual_std = np.sqrt(252) * np.std(daily_return)

# annual return
annual_return = 252 * np.mean(daily_return) - risk_free_rate

# annualized Sharpe ratio
sharpe = annual_return / annual_std

In [ ]:
sharpe